# IPCC LULUCF zonal stats summary spreadsheet by country (adm0)

Reads the combined flux and IPCC class area parquets from one zonal stats run (`flux_parquet_path`, `class_area_parquet_path` in the User Inputs cell) and writes one Excel workbook with two tabs:

- **flux**: `value` and `area_ha` summed by country, flux source, gas, IPCC classes, primary forest, year and analysis layer. `area_ha` is the area of rows with non-zero flux for that layer, not total land area.
- **ipcc_class_area**: total area (all pixels, whether or not they have flux) by country, IPCC class, change and summary (codes and names) and year. IPCC node code is summed out of both tabs. Use this for total country/class areas.

Before the tabs are summed, the **IPCC class reassignment rules** from `1_global_LULUCF_vegetation_IPCC_summary_spreadsheet.ipynb` are applied to both: rows whose `IPCC_class_name` is `Unassigned` get an IPCC class, change and summary from their vegetation model land state (`land_state_broad_class`, `tall_veg_type`). Then the **`ipcc_rules.csv` reassignment table** (`ipcc_rules_path`) is applied to the raw rows of both tables: a row whose IPCC codes, class name, primary forest flag and land state match a rule gets the rule's `*_updated` IPCC summary, node code, change, class and class name (same as `apply_ipcc_rules` in the old `IPCC_LULUCF_summary_spreadsheet` notebook, but on the raw parquet rows instead of the global tile summary). Rule key columns that the parquets don't have (e.g. `land_state_detailed_class`, `land_state_meaning`) are dropped from the match; rules that disagree once those are dropped are skipped with a warning and their rows keep their original codes. `IPCC_change_name`, `IPCC_summary_name` and `IPCC_node_code_name` are updated to match the new codes. Land state and primary forest columns are then summed out of the class area tab. If `WRITE_UPDATED_PARQUETS` is True, the reassigned raw tables are also written to `output_dir` as `..._<description>_updated_<timestamp>.parquet`. The workbook is written to `output_dir` too.

`IPCC_initial` is the first digit of `IPCC_change` (the IPCC class at the start of the change interval; 0 for Unassigned). It is calculated after the reassignment rules and is in both tabs.

`flux_source` is taken from the analysis layer name prefix: `LULUCF_`, `SOC_` (mineral soil), `organic_soil_` (also `burned_`/`drained_` in older outputs), and for vegetation `gross_emissions__`, `gross_removals__` and `net_flux__` (current zonal stats outputs) or `veg_` (older outputs).

The flux parquet needs the IPCC contextual layers (the zonal stats run needs `-cl` to include `IPCC_class IPCC_node IPCC_change IPCC_summary`); the notebook stops with a message if they're missing. Set `repo_root` below so `src.utilities` can be imported.

In [1]:
from pathlib import Path
import gc
import importlib.util
import re
import sys

import pandas as pd

# Root of the AFOLU flux model repo for the shared utilities import.
repo_root = Path(r"/mnt/c/GIS/git/AFOLU_GHG_flux_model")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.utilities import constants_and_names as cn
from src.utilities import universal_utilities as uu

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.3f}".format)

### User Inputs

In [2]:
# Combined zonal stats outputs for the run
flux_parquet_path = Path(r"/mnt/c/gis/afolu_model/land_use/zonal_statistics/EU27/raw/ipcc_flux/ipcc_lulucf_zonal_stats_v1_0_0_EU27_20261020_22_41_27.parquet")
class_area_parquet_path = Path(r"/mnt/c/gis/afolu_model/land_use/zonal_statistics/EU27/raw/ipcc_class_area/ipcc_class_area_by_country_v1_0_0_EU27_20261019_18_47_35.parquet")

# IPCC reassignment rules table (original condition columns + *_updated result columns)
ipcc_rules_path = Path(r"/mnt/c/gis/afolu_model/land_use/zonal_statistics/ipcc_rules.csv")

# The reassigned flux and class area parquets and the summary workbook are written here
output_dir = Path(r"/mnt/c/gis/afolu_model/land_use/zonal_statistics/EU27/reassigned")
output_dir.mkdir(parents=True, exist_ok=True)

# Also write the reassigned raw flux and class area tables to output_dir as parquet
WRITE_UPDATED_PARQUETS = True

# Description used in the output file names
zonal_stats_description = "EU27"

# Keep only the EU27 countries
FILTER_TO_EU27 = True

EU27_ISO3 = [
    "AUT", "BEL", "BGR", "HRV", "CYP", "CZE", "DNK", "EST", "FIN", "FRA", "DEU", "GRC", "HUN", "IRL",
    "ITA", "LVA", "LTU", "LUX", "MLT", "NLD", "POL", "PRT", "ROU", "SVK", "SVN", "ESP", "SWE",
]

# Countries to keep when FILTER_TO_EU27 is False (ISO3). Set to None to keep every country.
other_countries = None

countries = EU27_ISO3 if FILTER_TO_EU27 else other_countries

for label, p in [("Flux parquet", flux_parquet_path), ("IPCC class area parquet", class_area_parquet_path), ("IPCC rules", ipcc_rules_path)]:
    print(f"{label}: {p} | exists: {p.exists()}")
print("Output folder:", output_dir)
print("Zonal stats description:", zonal_stats_description)
print("Countries:", "EU27" if FILTER_TO_EU27 else (countries if countries else "all"))


Flux parquet: /mnt/c/gis/afolu_model/land_use/zonal_statistics/EU27/raw/ipcc_flux/ipcc_lulucf_zonal_stats_v1_0_0_EU27_20261020_22_41_27.parquet | exists: True
IPCC class area parquet: /mnt/c/gis/afolu_model/land_use/zonal_statistics/EU27/raw/ipcc_class_area/ipcc_class_area_by_country_v1_0_0_EU27_20261019_18_47_35.parquet | exists: True
IPCC rules: /mnt/c/gis/afolu_model/land_use/zonal_statistics/ipcc_rules.csv | exists: True
Output folder: /mnt/c/gis/afolu_model/land_use/zonal_statistics/EU27/reassigned
Zonal stats description: EU27
Countries: EU27


### IPCC class reassignment rules

In [3]:
numeric_to_ipcc_class = {
    0: "Unassigned",
    1: "Settlements",
    2: "Cropland",
    3: "Forest Land",
    4: "Grassland",
    5: "Wetlands",
    6: "Other Land (Bare)",
    7: "Other Land (Water)",
    8: "Other Land (Snow/Ice)",
}

assignment_rules = [
    {
        "name": "crop_to_cropland",
        "match": {"land_state_broad_class": "crop"},
        "fill": {"IPCC_summary": 22, "IPCC_change": 22, "IPCC_class": 2, "IPCC_class_name": "Cropland"},
    },
    {
        "name": "short_veg_to_grassland",
        "match": {"land_state_broad_class": "short_veg"},
        "fill": {"IPCC_summary": 44, "IPCC_change": 44, "IPCC_class": 4, "IPCC_class_name": "Grassland"},
    },
    {
        "name": "mangrove_to_forest",
        "match": {"land_state_broad_class": "tree", "tall_veg_type": "mangrove"},
        "fill": {"IPCC_summary": 33, "IPCC_change": 33, "IPCC_class": 3, "IPCC_class_name": "Forest Land"},
    },
    {
        "name": "natural_tree_cover_to_forest",
        "match": {"land_state_broad_class": "tree", "tall_veg_type": "natural_tree_cover"},
        "fill": {"IPCC_summary": 33, "IPCC_change": 33, "IPCC_class": 3, "IPCC_class_name": "Forest Land"},
    },
    {
        "name": "oil_palm_to_cropland",
        "match": {"land_state_broad_class": "tree", "tall_veg_type": "oil_palm"},
        "fill": {"IPCC_summary": 22, "IPCC_change": 22, "IPCC_class": 2, "IPCC_class_name": "Cropland"},
    },
    {
        "name": "non_oil_palm_planted_trees_to_cropland",
        "match": {"land_state_broad_class": "tree", "tall_veg_type": "non_oil_palm_planted_trees"},
        "fill": {"IPCC_summary": 22, "IPCC_change": 22, "IPCC_class": 2, "IPCC_class_name": "Cropland"},
    },
    {
        "name": "trees_in_other_land_covers_to_forest",
        "match": {"land_state_broad_class": "tree", "tall_veg_type": "trees_in_other_land_covers"},
        "fill": {"IPCC_summary": 33, "IPCC_change": 33, "IPCC_class": 3, "IPCC_class_name": "Forest Land"},
    },
    {
        "name": "no_flux_stays_unassigned",
        "match": {"land_state_broad_class": "no_flux"},
        "fill": {"IPCC_summary": 0, "IPCC_change": 0, "IPCC_class": 0, "IPCC_class_name": "Unassigned"},
    },
]

In [4]:
# Flux tab: columns summed over (only those present in the run are used)
FLUX_GROUP_COLS = [
    cn.adm0_pattern,
    "country_name",
    "flux_source",
    "gas",
    "IPCC_summary",
    "IPCC_summary_name",
    #"IPCC_node_code",
    #"IPCC_node_code_name",
    "year",
    "IPCC_change",
    "IPCC_change_name",
    "IPCC_initial",
    "IPCC_class",
    "IPCC_class_name",
    "starting_composite_primary_forest",
    "analysis_layer",
]
VALUE_COLS = ["value", "area_ha"]

# ipcc_class_area tab: IPCC class, change and summary columns from the class area parquets, plus IPCC_initial.
# IPCC node code, region, primary forest, land state and tile columns are summed out.
CLASS_AREA_GROUP_COLS = [
    cn.adm0_pattern, "country_name",
    "IPCC_class", "IPCC_class_name",
    "IPCC_change", "IPCC_change_name",
    "IPCC_initial",
    "IPCC_summary", "IPCC_summary_name",
    #"IPCC_node_code", "IPCC_node_code_name",
    "year",
]
CLASS_AREA_COLS = CLASS_AREA_GROUP_COLS + ["area_ha"]

# Land state columns the reassignment rules match on
RULE_MATCH_COLS = sorted({col for rule in assignment_rules for col in rule["match"]})

# Analysis layer name prefix -> flux source
FLUX_SOURCE_PREFIXES = {
    "LULUCF_": "LULUCF",
    "veg_": "vegetation",          # Vegetation layer names in older outputs
    "gross_emissions_": "vegetation",   # Vegetation layer names in current outputs (no veg_ prefix)
    "gross_removals_": "vegetation",
    "net_flux_": "vegetation",
    "SOC_": "mineral_soil",
    "organic_soil_": "organic_soil",
    "burned_": "organic_soil",     # Organic soil layer names in outputs made before the organic_soil_ prefix
    "drained_": "organic_soil",
}

EXCEL_MAX_ROWS = 1_048_575   # Excel's row limit, minus the header row


def log(message):
    print(f"{uu.timestr()}: {message}", flush=True)


# Columns the input parquets need. The IPCC columns come from the IPCC contextual layers in the zonal stats run, and
# IPCC_node_code is a key of the ipcc_rules.csv reassignment.
IPCC_COLS = ["IPCC_summary", "IPCC_node_code", "IPCC_change", "IPCC_class", "IPCC_class_name"]
FLUX_REQUIRED_COLS = [cn.adm0_pattern, "country_name", "year", "analysis_layer", "value", "area_ha"] + IPCC_COLS + RULE_MATCH_COLS
CLASS_AREA_REQUIRED_COLS = [cn.adm0_pattern, "country_name", "year", "area_ha"] + IPCC_COLS + RULE_MATCH_COLS


def read_input(path, table_name, required_cols):
    """Reads one combined zonal stats parquet and checks it has the columns this notebook needs."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"{table_name} parquet not found: {path}. Set the path in the User Inputs cell.")
    df = pd.read_parquet(path)
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        hint = (" The zonal stats run needs the IPCC contextual layers (-cl ... IPCC_class IPCC_node IPCC_change IPCC_summary)."
                if any(c in IPCC_COLS for c in missing) else "")
        raise ValueError(f"{table_name} parquet {path.name} has no {missing} columns.{hint}")
    log(f"Read {path.name}: {len(df):,} rows")
    return df


def flux_source(layer):
    for prefix, source in FLUX_SOURCE_PREFIXES.items():
        if layer.startswith(prefix):
            return source
    return "other"


def keep_countries(df, table_name):
    if countries:
        df = df[df[cn.adm0_pattern].isin(countries)].copy()
        missing = sorted(set(countries) - set(df[cn.adm0_pattern].unique()))
        if missing:
            log(f"WARNING: {table_name} has no rows for {missing}")
    return df


def apply_assignment_rule(df, table_name, rules=assignment_rules):
    """IPCC class reassignment from the global summary notebook: fills IPCC class, change and summary for Unassigned rows
    from their land state. Also updates IPCC_change_name and IPCC_summary_name for the rows it changes.
    Returns the df and a report of rows (and area) changed by each rule."""
    missing = [c for c in RULE_MATCH_COLS + ["IPCC_class_name"] if c not in df.columns]
    if missing:
        raise ValueError(f"{table_name} has no {missing} columns, so the IPCC class reassignment rules can't be applied")

    df["IPCC_class_name"] = df["IPCC_class_name"].fillna("Unassigned")
    base_unassigned = df["IPCC_class_name"].astype(str).eq("Unassigned")
    reassigned = pd.Series(False, index=df.index)
    report = []

    for rule in rules:
        mask = base_unassigned.copy()

        for col, val in rule["match"].items():
            mask &= df[col].astype(str).eq(str(val))

        if mask.any():
            for fill_col, fill_val in rule["fill"].items():
                df.loc[mask, fill_col] = fill_val
            reassigned |= mask

        report.append({"table": table_name, "rule": rule["name"], "rows": int(mask.sum()),
                       "area_ha": float(df.loc[mask, "area_ha"].sum())})

    for col in ["IPCC_summary", "IPCC_change", "IPCC_class"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("int64")

    class_map_mask = df["IPCC_class_name"].isna() | df["IPCC_class_name"].astype(str).eq("")
    if "IPCC_class" in df.columns:
        df.loc[class_map_mask, "IPCC_class_name"] = df.loc[class_map_mask, "IPCC_class"].map(numeric_to_ipcc_class).fillna("Unassigned")

    # Change and summary names that match the reassigned codes
    for col in ["IPCC_change", "IPCC_summary"]:
        if f"{col}_name" in df.columns:
            df.loc[reassigned, f"{col}_name"] = df.loc[reassigned, col].map(cn.numeric_to_ipcc_change).fillna("Unassigned")

    log(f"{table_name}: reassignment rules changed {int(reassigned.sum()):,} of {int(base_unassigned.sum()):,} Unassigned rows")
    return df, pd.DataFrame(report)


# ipcc_rules.csv: original condition columns (keys) and the corrected IPCC columns (<col>_updated)
IPCC_RULE_KEY_COLS = [
    "IPCC_summary", "IPCC_node_code", "IPCC_change", "IPCC_class", "IPCC_class_name",
    "starting_composite_primary_forest", "land_state_broad_class", "tall_veg_type",
    "land_state_detailed_class", "land_state_meaning",
]
IPCC_RULE_OUTPUT_COLS = ["IPCC_summary", "IPCC_node_code", "IPCC_change", "IPCC_class", "IPCC_class_name"]
IPCC_RULE_UPDATED_COLS = [f"{c}_updated" for c in IPCC_RULE_OUTPUT_COLS]


def load_ipcc_rules(rules_path, available_cols):
    """Reads ipcc_rules.csv. Key columns the data doesn't have are dropped from the match; rules that then disagree
    with each other are removed (rows with those keys keep their original codes). Returns (rules, key columns)."""
    if not Path(rules_path).exists():
        raise FileNotFoundError(f"IPCC rules not found: {rules_path}. Set ipcc_rules_path in the user inputs cell.")
    rules = pd.read_csv(rules_path)
    missing = [c for c in IPCC_RULE_KEY_COLS + IPCC_RULE_UPDATED_COLS if c not in rules.columns]
    if missing:
        raise ValueError(f"{Path(rules_path).name} is missing required columns: {missing}")

    keys = [c for c in IPCC_RULE_KEY_COLS if c in available_cols]
    dropped = [c for c in IPCC_RULE_KEY_COLS if c not in keys]
    if dropped:
        log(f"WARNING: data has no {dropped} columns; IPCC rules are matched on {keys}")
    rules = rules[keys + IPCC_RULE_UPDATED_COLS].drop_duplicates().reset_index(drop=True)

    conflicts = rules.groupby(keys, dropna=False)[IPCC_RULE_UPDATED_COLS].nunique(dropna=False).max(axis=1).gt(1)
    if conflicts.any():
        bad = conflicts[conflicts].reset_index()[keys]
        log(f"WARNING: {len(bad)} rule conditions give different results without {dropped}; "
            f"rows matching them are left unchanged:")
        display(bad)
        rules = rules.merge(bad.assign(_conflict=True), on=keys, how="left")
        rules = rules[rules["_conflict"].isna()].drop(columns="_conflict").reset_index(drop=True)
    log(f"Loaded {len(rules):,} IPCC rules from {Path(rules_path).name}")
    return rules, keys


def apply_ipcc_rules(df, table_name, rules, keys):
    """Applies ipcc_rules.csv to the raw rows: rows whose key columns match a rule get its *_updated IPCC summary, node
    code, change, class and class name, and the matching change/summary/node code names. Returns df and a report row."""
    left = df[keys].copy()
    right = rules.copy()
    for c in keys:   # same dtype on both sides of the merge
        if pd.api.types.is_numeric_dtype(df[c]):
            left[c] = pd.to_numeric(left[c]).astype("int64")
            right[c] = pd.to_numeric(right[c]).astype("int64")
        else:
            left[c] = left[c].astype(str)
            right[c] = right[c].astype(str)
    merged = left.merge(right, on=keys, how="left", validate="many_to_one")
    matched = merged[IPCC_RULE_UPDATED_COLS[0]].notna().to_numpy()

    for col in ["IPCC_summary", "IPCC_node_code", "IPCC_change", "IPCC_class"]:
        df[col] = pd.to_numeric(df[col], errors="raise").astype("int64")
    for col, updated in zip(IPCC_RULE_OUTPUT_COLS, IPCC_RULE_UPDATED_COLS):
        df.loc[matched, col] = merged.loc[matched, updated].to_numpy()
    for col in ["IPCC_summary", "IPCC_node_code", "IPCC_change", "IPCC_class"]:
        df[col] = df[col].astype("int64")

    # Names that match the updated codes
    for col, mapping in [("IPCC_change", cn.numeric_to_ipcc_change), ("IPCC_summary", cn.numeric_to_ipcc_change),
                         ("IPCC_node_code", cn.numeric_to_ipcc_node_code)]:
        if f"{col}_name" in df.columns:
            df.loc[matched, f"{col}_name"] = df.loc[matched, col].map(mapping).fillna("Unassigned")

    log(f"{table_name}: ipcc_rules.csv changed {int(matched.sum()):,} of {len(df):,} rows")
    report = pd.DataFrame([{"table": table_name, "rule": "ipcc_rules.csv", "rows": int(matched.sum()),
                            "area_ha": float(df.loc[matched, "area_ha"].sum())}])
    return df, report


def add_ipcc_initial(df):
    """IPCC_initial: first digit of IPCC_change (codes are 10 x initial class + final class; 0 is Unassigned)."""
    df["IPCC_initial"] = (df["IPCC_change"].astype("int64") // 10).astype("int64")
    return df


def summarize_flux(df, source, updated_parquet=None):
    df = keep_countries(df, "flux")
    log(f"Flux: {len(df):,} input rows for {sorted(df[cn.adm0_pattern].unique())}")
    if df.empty:
        raise ValueError(f"No flux rows for {countries} in {source}")

    df["flux_source"] = df["analysis_layer"].map(flux_source)
    unknown = sorted(df.loc[df["flux_source"] == "other", "analysis_layer"].unique())
    if unknown:
        log(f"WARNING: no flux source for {unknown}; labelled 'other'")

    # Values are float32 in the parquet; sum in float64 so totals don't lose precision
    df[VALUE_COLS] = df[VALUE_COLS].astype("float64")
    input_totals = df.groupby("analysis_layer")[VALUE_COLS].sum()

    df, rule_report = apply_assignment_rule(df, "flux")
    df, csv_report = apply_ipcc_rules(df, "flux", ipcc_rules, ipcc_rule_keys)
    rule_report = pd.concat([rule_report, csv_report], ignore_index=True)
    df = add_ipcc_initial(df)
    if updated_parquet is not None:
        df.to_parquet(updated_parquet, index=False)
        log(f"Saved {updated_parquet}")

    group_cols = [c for c in FLUX_GROUP_COLS if c in df.columns]
    summary = (
        df.groupby(group_cols, dropna=False, observed=True, as_index=False)[VALUE_COLS]
          .sum()
          .sort_values(group_cols)
          .reset_index(drop=True)
    )
    del df
    gc.collect()

    # Reassigning and dropping detail columns only moves and merges rows, so totals per layer must be unchanged
    output_totals = summary.groupby("analysis_layer")[VALUE_COLS].sum()
    diff = (output_totals - input_totals).abs().max().max()
    log(f"Flux: {len(summary):,} output rows; max total difference vs input = {diff:,.6f}")

    return summary, rule_report


def summarize_class_area(df, source, updated_parquet=None):
    df = keep_countries(df, "IPCC class area")
    log(f"IPCC class area: {len(df):,} input rows for {sorted(df[cn.adm0_pattern].unique())}")
    if df.empty:
        raise ValueError(f"No IPCC class area rows for {countries} in {source}")

    needed = [c for c in CLASS_AREA_COLS + RULE_MATCH_COLS if c not in df.columns and c != "IPCC_initial"]
    if needed:
        raise ValueError(f"IPCC class area data ({source}) has no {needed} columns. It needs to come from the zonal stats "
                         f"script version that writes IPCC change/summary/node and land state to the class area tables.")

    df["area_ha"] = df["area_ha"].astype("float64")
    input_totals = df.groupby([cn.adm0_pattern, "year"])["area_ha"].sum()

    df, rule_report = apply_assignment_rule(df, "ipcc_class_area")
    df, csv_report = apply_ipcc_rules(df, "ipcc_class_area", ipcc_rules, ipcc_rule_keys)
    rule_report = pd.concat([rule_report, csv_report], ignore_index=True)
    df = add_ipcc_initial(df)
    if updated_parquet is not None:
        df.to_parquet(updated_parquet, index=False)
        log(f"Saved {updated_parquet}")

    # Sorted by whichever of these columns are in CLASS_AREA_GROUP_COLS (so columns can be commented out of it)
    sort_cols = [c for c in [cn.adm0_pattern, "year", "IPCC_class", "IPCC_change", "IPCC_summary", "IPCC_node_code"]
                 if c in CLASS_AREA_GROUP_COLS]
    summary = (
        df.groupby(CLASS_AREA_GROUP_COLS, dropna=False, observed=True, as_index=False)["area_ha"]
          .sum()
          .sort_values(sort_cols)
          .reset_index(drop=True)
    )[CLASS_AREA_COLS]

    diff = (summary.groupby([cn.adm0_pattern, "year"])["area_ha"].sum() - input_totals).abs().max()
    log(f"IPCC class area: {len(summary):,} output rows; max country-year total difference vs input = {diff:,.6f} ha")

    return summary, rule_report


def write_workbook(tabs, path):
    """Writes each dataframe to its own tab: Arial, bold frozen header row, filters, number formats, column widths."""
    for name, df in tabs.items():
        if len(df) > EXCEL_MAX_ROWS:
            raise ValueError(f"{name} has {len(df):,} rows, more than Excel allows ({EXCEL_MAX_ROWS:,}). "
                             f"Write it to CSV or parquet instead, or keep fewer countries.")

    number_formats = {"value": "#,##0.00", "area_ha": "#,##0.0"}

    # xlsxwriter is much faster for large tabs; openpyxl is the fallback
    if importlib.util.find_spec("xlsxwriter"):
        with pd.ExcelWriter(path, engine="xlsxwriter") as writer:
            book = writer.book
            book.formats[0].set_font_name("Arial")
            header_fmt = book.add_format({"bold": True, "font_name": "Arial", "bottom": 1, "text_wrap": True, "valign": "top"})
            for name, df in tabs.items():
                df.to_excel(writer, sheet_name=name, index=False)
                ws = writer.sheets[name]
                for i, col in enumerate(df.columns):
                    ws.write(0, i, col, header_fmt)
                    width = min(max(len(str(col)), int(df[col].astype(str).str.len().max() or 0)) + 2, 60)
                    fmt = book.add_format({"font_name": "Arial", "num_format": number_formats[col]}) if col in number_formats else None
                    ws.set_column(i, i, width, fmt)
                ws.freeze_panes(1, 0)
                ws.autofilter(0, 0, len(df), len(df.columns) - 1)
    else:
        from openpyxl.styles import Font
        from openpyxl.utils import get_column_letter
        with pd.ExcelWriter(path, engine="openpyxl") as writer:
            for name, df in tabs.items():
                df.to_excel(writer, sheet_name=name, index=False)
                ws = writer.sheets[name]
                for i, col in enumerate(df.columns, start=1):
                    letter = get_column_letter(i)
                    width = min(max(len(str(col)), int(df[col].astype(str).str.len().max() or 0)) + 2, 60)
                    ws.column_dimensions[letter].width = width
                    for (cell,) in ws.iter_rows(min_col=i, max_col=i):
                        cell.font = Font(name="Arial", bold=(cell.row == 1))
                        if cell.row > 1 and col in number_formats:
                            cell.number_format = number_formats[col]
                ws.freeze_panes = "A2"
                ws.auto_filter.ref = ws.dimensions

    log(f"Saved {path}")

## Summarize and write the workbook

In [5]:
flux_df = read_input(flux_parquet_path, "flux", FLUX_REQUIRED_COLS)
class_area_df = read_input(class_area_parquet_path, "IPCC class area", CLASS_AREA_REQUIRED_COLS)
flux_source_desc, class_area_source_desc = flux_parquet_path.name, class_area_parquet_path.name

timestamp = uu.timestr()
updated_desc = f"{zonal_stats_description}_updated"
flux_parquet = output_dir / f"ipcc_lulucf_zonal_stats_{updated_desc}_{timestamp}.parquet" if WRITE_UPDATED_PARQUETS else None
class_area_parquet = output_dir / f"ipcc_class_area_by_country_{updated_desc}_{timestamp}.parquet" if WRITE_UPDATED_PARQUETS else None

# Rules are matched on the key columns both tables have
ipcc_rules, ipcc_rule_keys = load_ipcc_rules(ipcc_rules_path, set(flux_df.columns) & set(class_area_df.columns))

flux_summary, flux_rule_report = summarize_flux(flux_df, flux_source_desc, flux_parquet)
class_area, class_area_rule_report = summarize_class_area(class_area_df, class_area_source_desc, class_area_parquet)
del flux_df, class_area_df
gc.collect()

country_label = "EU27" if FILTER_TO_EU27 else ("_".join(countries) if countries else "all_countries")
workbook_path = Path(output_dir) / f"IPCC_LULUCF_zonal_stats_summary__{zonal_stats_description}__{country_label}__{timestamp}.xlsx"

write_workbook({"flux": flux_summary, "ipcc_class_area": class_area}, workbook_path)
workbook_path

20261021_01_57_56: Read ipcc_lulucf_zonal_stats_v1_0_0_EU27_20261020_22_41_27.parquet: 3,942,832 rows
20261021_01_57_57: Read ipcc_class_area_by_country_v1_0_0_EU27_20261019_18_47_35.parquet: 325,216 rows
20261021_01_57_57: WARNING: data has no ['land_state_detailed_class', 'land_state_meaning'] columns; IPCC rules are matched on ['IPCC_summary', 'IPCC_node_code', 'IPCC_change', 'IPCC_class', 'IPCC_class_name', 'starting_composite_primary_forest', 'land_state_broad_class', 'tall_veg_type']
20261021_01_57_57: WARNING: 6 rule conditions give different results without ['land_state_detailed_class', 'land_state_meaning']; rows matching them are left unchanged:


,IPCC_summary,IPCC_node_code,IPCC_change,IPCC_class,IPCC_class_name,starting_composite_primary_forest,land_state_broad_class,tall_veg_type
0,86,80,88,8,Other Land (Snow/Ice),0,tree,trees_in_other_land_covers
1,86,80,88,8,Other Land (Snow/Ice),1,tree,trees_in_other_land_covers
2,88,80,88,8,Other Land (Snow/Ice),0,tree,trees_in_other_land_covers
3,88,80,88,8,Other Land (Snow/Ice),1,tree,trees_in_other_land_covers
4,88,89,88,8,Other Land (Snow/Ice),0,short_veg,non_tall_vegetation
5,88,89,88,8,Other Land (Snow/Ice),1,short_veg,non_tall_vegetation


20261021_01_57_57: Loaded 362 IPCC rules from ipcc_rules.csv
20261021_01_57_58: Flux: 1,882,803 input rows for ['AUT', 'BEL', 'BGR', 'CYP', 'CZE', 'DEU', 'DNK', 'ESP', 'EST', 'FIN', 'FRA', 'GRC', 'HRV', 'HUN', 'IRL', 'ITA', 'LTU', 'LUX', 'LVA', 'MLT', 'NLD', 'POL', 'PRT', 'ROU', 'SVK', 'SVN', 'SWE']
20261021_01_58_00: flux: reassignment rules changed 925 of 925 Unassigned rows
20261021_01_58_01: flux: ipcc_rules.csv changed 50,582 of 1,882,803 rows
20261021_01_58_05: Saved /mnt/c/gis/afolu_model/land_use/zonal_statistics/EU27/reassigned/ipcc_lulucf_zonal_stats_EU27_updated_20261021_01_57_57.parquet
20261021_01_58_06: Flux: 94,209 output rows; max total difference vs input = 0.000000
20261021_01_58_06: IPCC class area: 146,459 input rows for ['AUT', 'BEL', 'BGR', 'CYP', 'CZE', 'DEU', 'DNK', 'ESP', 'EST', 'FIN', 'FRA', 'GRC', 'HRV', 'HUN', 'IRL', 'ITA', 'LTU', 'LUX', 'LVA', 'MLT', 'NLD', 'POL', 'PRT', 'ROU', 'SVK', 'SVN', 'SWE']
20261021_01_58_06: ipcc_class_area: reassignment rules chan

PosixPath('/mnt/c/gis/afolu_model/land_use/zonal_statistics/EU27/reassigned/IPCC_LULUCF_zonal_stats_summary__EU27__EU27__20261021_01_57_57.xlsx')

## QA

In [6]:
# Rows (and, for the class area tab, area) changed by each reassignment rule. The 'ipcc_rules.csv' row is the rules table,
# applied after the Unassigned rules. Flux area_ha is repeated for every analysis layer, so only the class area tab's area_ha is a real area.
display(pd.concat([flux_rule_report, class_area_rule_report], ignore_index=True))

,table,rule,rows,area_ha
0,flux,crop_to_cropland,0,0.000
1,flux,short_veg_to_grassland,151,50.660
2,flux,mangrove_to_forest,0,0.000
3,flux,natural_tree_cover_to_forest,86,11.113
4,flux,oil_palm_to_cropland,0,0.000
5,flux,non_oil_palm_planted_trees_to_cropland,357,36.219
6,flux,trees_in_other_land_covers_to_forest,106,12.145
7,flux,no_flux_stays_unassigned,225,"1,985.166"
8,flux,ipcc_rules.csv,50582,"282,287,805.685"
9,ipcc_class_area,crop_to_cropland,0,0.000


In [7]:
# Total IPCC class area per country and year (Mha), after reassignment. Compare with official country areas;
# GADM boundaries include inland water (IPCC class "Other Land (Water)").
display(
    class_area.assign(area_Mha=class_area["area_ha"] / 1_000_000)
              .pivot_table(index=["country_name", "year"], columns="IPCC_class_name", values="area_Mha", aggfunc="sum", fill_value=0)
              .assign(total_Mha=lambda d: d.sum(axis=1))
)

IPCC_class_name    Cropland  Forest Land  Grassland  Other Land (Bare)  \
country_name year                                                        
Austria      2016     0.976        3.370      1.544              0.111   
             2017     0.977        3.352      1.526              0.107   
             2018     0.978        3.334      1.522              0.108   
             2019     0.978        3.319      1.518              0.110   
             2020     0.972        3.303      1.507              0.113   
...                     ...          ...        ...                ...   
Sweden       2020     1.838       27.263      7.870              0.163   
             2021     1.836       27.158      7.844              0.164   
             2022     1.840       27.118      7.846              0.164   
             2023     1.838       27.052      7.828              0.164   
             2024     1.822       26.959      7.799              0.162   

IPCC_class_name    Other Land (Snow/Ice)  Settlements  Wetlands  total_Mha  
country_name year                                                           
Austria      2016                  0.066        2.236     0.083      8.386  
             2017                  0.066        2.265     0.083      8.377  
             2018                  0.066        2.289     0.083      8.381  
             2019                  0.066        2.307     0.083      8.383  
             2020                  0.064        2.340     0.082      8.382  
...                                  ...          ...       ...        ...  
Sweden       2020                  0.090        3.623     4.049     44.896  
             2021                  0.090        3.764     4.045     44.900  
             2022                  0.090        3.804     4.051     44.913  
             2023                  0.090        3.875     4.054     44.901  
             2024                  0.090        4.061     4.015     44.907  

[243 rows x 8 columns]

In [8]:
# Consistency: the second digit of IPCC_change is the class at the end of the interval, so it should equal IPCC_class.
# Area where it doesn't (excluding change code 0, Unassigned):
check = class_area[class_area["IPCC_change"] != 0]
mismatch = check[(check["IPCC_change"] % 10) != check["IPCC_class"]]
print(f"Area where IPCC_change doesn't end in IPCC_class: {mismatch['area_ha'].sum():,.1f} ha of {check['area_ha'].sum():,.1f} ha")
display(mismatch.groupby(["IPCC_change_name", "IPCC_class_name"])["area_ha"].sum().sort_values(ascending=False).head(20))

Area where IPCC_change doesn't end in IPCC_class: 0.0 ha of 3,709,285,461.7 ha


Series([], Name: area_ha, dtype: float64)

In [9]:
# Area by initial IPCC class and year (Mha)
display(
    class_area.assign(area_Mha=class_area["area_ha"] / 1_000_000)
              .pivot_table(index=["country_name", "IPCC_initial"], columns="year", values="area_Mha", aggfunc="sum", fill_value=0)
)

year                        2016   2017   2018   2019   2020   2021   2022  \
country_name IPCC_initial                                                    
Austria      1             2.174  2.238  2.265  2.289  2.308  2.341  2.386   
             2             0.983  0.976  0.977  0.978  0.978  0.972  0.969   
             3             3.410  3.370  3.351  3.334  3.319  3.304  3.280   
             4             1.559  1.536  1.530  1.524  1.518  1.506  1.489   
             5             0.083  0.083  0.083  0.083  0.083  0.083  0.083   
...                          ...    ...    ...    ...    ...    ...    ...   
Sweden       2             1.815  1.811  1.820  1.831  1.844  1.838  1.836   
             3            27.698 27.535 27.473 27.397 27.345 27.271 27.155   
             4             7.974  7.940  7.915  7.906  7.887  7.865  7.851   
             5             4.050  4.045  4.050  4.051  4.050  4.045  4.050   
             6             0.248  0.247  0.247  0.249  0.252  0.253  0.253   

year                        2023   2024  
country_name IPCC_initial                
Austria      1             2.395  2.430  
             2             0.970  0.966  
             3             3.273  3.256  
             4             1.485  1.473  
             5             0.083  0.082  
...                          ...    ...  
Sweden       2             1.840  1.838  
             3            27.108 27.090  
             4             7.843  7.828  
             5             4.054  4.015  
             6             0.253  0.252  

[171 rows x 9 columns]

In [10]:
# Flux totals by flux source and analysis layer
display(flux_summary.groupby(["flux_source", "analysis_layer"])[VALUE_COLS].sum())

value  \
flux_source  analysis_layer                                                       
mineral_soil SOC_net__mineral_soil_extent__0-30cm_MgCO2         755,073,492.274   
organic_soil organic_soil_burned_co2_Mg_CO2                       9,610,832.577   
             organic_soil_burned_total_Mg_CO2e                   11,370,484.032   
             organic_soil_drained_co2_Mg_CO2                  4,631,908,287.976   
             organic_soil_drained_total_Mg_CO2e               5,988,838,213.108   
vegetation   gross_emissions__all_C_pools__CO2_only__MgCO2    4,563,398,723.495   
             gross_emissions__all_C_pools__all_gases__MgCO2e  4,572,223,532.707   
             gross_removals__all_C_pools__MgCO2              -8,951,696,465.359   
             net_flux__all_C_pools__CO2_only__MgCO2          -4,388,297,724.655   
             net_flux__all_C_pools__all_gases__MgCO2e        -4,379,472,916.972   

                                                                       area_ha  
flux_source  analysis_layer                                                     
mineral_soil SOC_net__mineral_soil_extent__0-30cm_MgCO2      3,712,425,636.733  
organic_soil organic_soil_burned_co2_Mg_CO2                    976,503,816.203  
             organic_soil_burned_total_Mg_CO2e                 976,503,816.203  
             organic_soil_drained_co2_Mg_CO2                 2,600,246,956.733  
             organic_soil_drained_total_Mg_CO2e              2,600,246,956.733  
vegetation   gross_emissions__all_C_pools__CO2_only__MgCO2      31,902,882.183  
             gross_emissions__all_C_pools__all_gases__MgCO2e    34,738,992.971  
             gross_removals__all_C_pools__MgCO2              1,645,971,279.113  
             net_flux__all_C_pools__CO2_only__MgCO2          1,656,472,047.297  
             net_flux__all_C_pools__all_gases__MgCO2e        1,659,308,158.085

In [11]:
# Net flux by country and year for each flux source
net_layers = [l for l in flux_summary["analysis_layer"].unique() if "net" in l]
display(
    flux_summary[flux_summary["analysis_layer"].isin(net_layers)]
    .pivot_table(index=["country_name", "flux_source", "analysis_layer"], columns="year", values="value", aggfunc="sum")
)

year                                                                            2016  \
country_name flux_source  analysis_layer                                               
Austria      mineral_soil SOC_net__mineral_soil_extent__0-30cm_MgCO2   5,416,337.732   
             vegetation   net_flux__all_C_pools__CO2_only__MgCO2     -17,888,902.253   
                          net_flux__all_C_pools__all_gases__MgCO2e   -17,888,612.683   
Belgium      mineral_soil SOC_net__mineral_soil_extent__0-30cm_MgCO2    -868,789.068   
             vegetation   net_flux__all_C_pools__CO2_only__MgCO2      -6,950,135.249   
...                                                                              ...   
Spain        vegetation   net_flux__all_C_pools__CO2_only__MgCO2     -56,850,476.388   
                          net_flux__all_C_pools__all_gases__MgCO2e   -56,783,977.720   
Sweden       mineral_soil SOC_net__mineral_soil_extent__0-30cm_MgCO2   6,799,490.471   
             vegetation   net_flux__all_C_pools__CO2_only__MgCO2     -52,665,217.872   
                          net_flux__all_C_pools__all_gases__MgCO2e   -52,665,217.872   

year                                                                            2017  \
country_name flux_source  analysis_layer                                               
Austria      mineral_soil SOC_net__mineral_soil_extent__0-30cm_MgCO2   5,416,337.707   
             vegetation   net_flux__all_C_pools__CO2_only__MgCO2     -17,272,396.555   
                          net_flux__all_C_pools__all_gases__MgCO2e   -17,271,378.123   
Belgium      mineral_soil SOC_net__mineral_soil_extent__0-30cm_MgCO2    -868,789.108   
             vegetation   net_flux__all_C_pools__CO2_only__MgCO2      -6,931,002.482   
...                                                                              ...   
Spain        vegetation   net_flux__all_C_pools__CO2_only__MgCO2     -59,885,144.099   
                          net_flux__all_C_pools__all_gases__MgCO2e   -59,687,365.274   
Sweden       mineral_soil SOC_net__mineral_soil_extent__0-30cm_MgCO2   6,799,490.322   
             vegetation   net_flux__all_C_pools__CO2_only__MgCO2     -46,372,076.710   
                          net_flux__all_C_pools__all_gases__MgCO2e   -46,372,076.710   

year                                                                            2018  \
country_name flux_source  analysis_layer                                               
Austria      mineral_soil SOC_net__mineral_soil_extent__0-30cm_MgCO2   5,416,337.901   
             vegetation   net_flux__all_C_pools__CO2_only__MgCO2     -12,649,110.172   
                          net_flux__all_C_pools__all_gases__MgCO2e   -12,646,941.550   
Belgium      mineral_soil SOC_net__mineral_soil_extent__0-30cm_MgCO2    -868,789.080   
             vegetation   net_flux__all_C_pools__CO2_only__MgCO2      -6,775,977.223   
...                                                                              ...   
Spain        vegetation   net_flux__all_C_pools__CO2_only__MgCO2     -53,062,997.684   
                          net_flux__all_C_pools__all_gases__MgCO2e   -53,045,305.181   
Sweden       mineral_soil SOC_net__mineral_soil_extent__0-30cm_MgCO2   6,799,490.292   
             vegetation   net_flux__all_C_pools__CO2_only__MgCO2     -42,899,798.455   
                          net_flux__all_C_pools__all_gases__MgCO2e   -42,865,835.264   

year                                                                            2019  \
country_name flux_source  analysis_layer                                               
Austria      mineral_soil SOC_net__mineral_soil_extent__0-30cm_MgCO2   5,416,337.869   
             vegetation   net_flux__all_C_pools__CO2_only__MgCO2      -6,801,197.598   
                          net_flux__all_C_pools__all_gases__MgCO2e    -6,801,043.113   
Belgium      mineral_soil SOC_net__mineral_soil_extent__0-30cm_MgCO2    -868,789.101   
             vegetation   net_flux__a

In [12]:
class_area.head(10)

,adm0,country_name,IPCC_class,IPCC_class_name,IPCC_change,IPCC_change_name,IPCC_initial,IPCC_summary,IPCC_summary_name,year,area_ha
0,AUT,Austria,1,Settlements,11,Settlements remaining Settlements,1,11,Settlements remaining Settlements,2016,"2,173,697.447"
1,AUT,Austria,1,Settlements,21,Cropland to Settlements,2,21,Cropland to Settlements,2016,"7,651.023"
2,AUT,Austria,1,Settlements,31,Forest Land to Settlements,3,31,Forest Land to Settlements,2016,"40,049.628"
3,AUT,Austria,1,Settlements,41,Grassland to Settlements,4,41,Grassland to Settlements,2016,"14,226.046"
4,AUT,Austria,1,Settlements,51,Wetlands to Settlements,5,51,Wetlands to Settlements,2016,375.917
5,AUT,Austria,1,Settlements,61,Other Land (Bare) to Settlements,6,61,Other Land (Bare) to Settlements,2016,35.930
6,AUT,Austria,2,Cropland,22,Cropland remaining Cropland,2,21,Cropland to Settlements,2016,"47,128.376"
7,AUT,Austria,2,Cropland,22,Cropland remaining Cropland,2,22,Cropland remaining Cropland,2016,"928,540.002"
8,AUT,Austria,2,Cropland,32,Forest Land to Cropland,3,32,Forest Land to Cropland,2016,87.093
9,AUT,Austria,2,Cropland,42,Grassland to Cropland,4,42,Grassland to Cropland,2016,180.622
